<a href="https://colab.research.google.com/github/Bryan-Kweku-Wilson/Machine-Learning-Projects/blob/main/EcoCell_RL_06_RL_Environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

network_df = pd.read_csv("ecocell_network.csv")

traffic_df = pd.read_csv("ecocell_energy_qos.csv")

traffic_df["timestamp"] = pd.to_datetime(
    traffic_df["timestamp"]
)

network_df = network_df.sort_values(
    "bs_id"
).reset_index(drop=True)

traffic_df = traffic_df.sort_values(
    ["timestamp", "bs_id"]
).reset_index(drop=True)

print("Network shape:", network_df.shape)
print("Traffic shape:", traffic_df.shape)

print("Number of base stations:", network_df["bs_id"].nunique())
print("Number of timestamps:", traffic_df["timestamp"].nunique())

print("\nNetwork columns:")
print(network_df.columns.tolist())

print("\nTraffic columns:")
print(traffic_df.columns.tolist())

Network shape: (50, 7)
Traffic shape: (36000, 22)
Number of base stations: 50
Number of timestamps: 720

Network columns:
['bs_id', 'x', 'y', 'cell_type', 'capacity_mbps', 'coverage_radius_km', 'baseline_power_w']

Traffic columns:
['timestamp', 'bs_id', 'cell_type', 'traffic_mbps', 'active_users', 'capacity_mbps', 'utilization', 'throughput_mbps', 'packet_loss_rate', 'latency_ms', 'traffic_spike', 'idle_power_w', 'baseline_power_w', 'active_power_w', 'light_sleep_power_w', 'deep_sleep_power_w', 'bs_active_energy_kwh', 'light_sleep_energy_kwh', 'deep_sleep_energy_kwh', 'light_sleep_savings_kwh', 'deep_sleep_savings_kwh', 'network_active_energy_kwh']


Reconstruct the network graph

In [14]:
G = nx.Graph()

for _, row in network_df.iterrows():
    G.add_node(row["bs_id"])

for _, row in network_df.iterrows():
    bs_id = row["bs_id"]
    x = row["x"]
    y = row["y"]

    right_neighbor = network_df[
        (network_df["x"] == x + 1) &
        (network_df["y"] == y)
    ]

    upper_neighbor = network_df[
        (network_df["x"] == x) &
        (network_df["y"] == y + 1)
    ]

    if not right_neighbor.empty:
        G.add_edge(
            bs_id,
            right_neighbor.iloc[0]["bs_id"]
        )

    if not upper_neighbor.empty:
        G.add_edge(
            bs_id,
            upper_neighbor.iloc[0]["bs_id"]
        )

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

Graph nodes: 50
Graph edges: 85


In [15]:
node_ids = network_df["bs_id"].tolist()

node_to_idx = {
    bs_id: idx
    for idx, bs_id in enumerate(node_ids)
}

neighbors = {
    node_to_idx[bs_id]: [
        node_to_idx[neighbor]
        for neighbor in G.neighbors(bs_id)
    ]
    for bs_id in node_ids
}

print("BS00 index:", node_to_idx["BS00"])

print(
    "BS00 neighbors:",
    [
        node_ids[idx]
        for idx in neighbors[node_to_idx["BS00"]]
    ]
)

BS00 index: 0
BS00 neighbors: ['BS01', 'BS10']


In [16]:
%pip install -q gymnasium

In [17]:
import gymnasium as gym

from gymnasium import spaces

print("Gymnasium version:", gym.__version__)

Gymnasium version: 1.3.0


Implement the environment

In [18]:
class EcoCellEnv(gym.Env):

    metadata = {"render_modes": []}

    def __init__(self, traffic_df, network_df, neighbors):
        super().__init__()

        self.node_ids = network_df["bs_id"].tolist()
        self.num_nodes = len(self.node_ids)

        # Arrange every measurement as [time, base station].
        self.timestamps = sorted(
            traffic_df["timestamp"].unique()
        )

        self.data = {}

        columns_to_load = [
            "traffic_mbps",
            "active_users",
            "capacity_mbps",
            "baseline_power_w",
            "idle_power_w",
            "packet_loss_rate",
            "latency_ms"
        ]

        for column in columns_to_load:
            pivot = traffic_df.pivot(
                index="timestamp",
                columns="bs_id",
                values=column
            )

            pivot = pivot.reindex(
                index=self.timestamps,
                columns=self.node_ids
            )

            self.data[column] = pivot.to_numpy(
                dtype=np.float32
            )

        self.neighbors = neighbors

        # 0 = active, 1 = light sleep, 2 = deep sleep.
        self.action_space = spaces.MultiDiscrete(
            np.full(self.num_nodes, 3, dtype=np.int64)
        )

        # Six observable features per base station.
        self.num_features = 6

        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_nodes * self.num_features,),
            dtype=np.float32
        )

        self.power_state = np.zeros(
            self.num_nodes,
            dtype=np.int64
        )

        self.t = 0

    def _get_observation(self):

        traffic = self.data["traffic_mbps"][self.t]
        capacity = self.data["capacity_mbps"][self.t]
        users = self.data["active_users"][self.t]

        packet_loss = self.data["packet_loss_rate"][self.t]
        latency = self.data["latency_ms"][self.t]

        utilization = traffic / np.maximum(capacity, 1e-6)

        features = np.column_stack([
            np.clip(traffic / np.maximum(capacity, 1e-6) / 3, 0, 1),
            np.clip(users / 500, 0, 1),
            np.clip(utilization / 3, 0, 1),
            np.clip(packet_loss / 0.10, 0, 1),
            np.clip(latency / 100, 0, 1),
            self.power_state / 2.0
        ])

        return features.astype(np.float32).flatten()

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.t = 0

        # Begin with every base station active.
        self.power_state = np.zeros(
            self.num_nodes,
            dtype=np.int64
        )

        observation = self._get_observation()

        return observation, {}

    def step(self, action):

        action = np.asarray(action, dtype=np.int64).copy()

        # Current-hour traffic and network properties.
        traffic = self.data["traffic_mbps"][self.t]
        capacity = self.data["capacity_mbps"][self.t]
        baseline_power = self.data["baseline_power_w"][self.t]
        idle_power = self.data["idle_power_w"][self.t]

        # --------------------------------------------------
        # 1. Safety check: ensure sleeping cells have
        #    at least one active neighbor.
        # --------------------------------------------------

        if np.all(action != 0):
            # If every cell is sleeping, keep the
            # highest-demand cell active.
            action[np.argmax(traffic)] = 0

        for node_idx in range(self.num_nodes):

            if action[node_idx] == 0:
                continue

            active_neighbors = [
                neighbor
                for neighbor in self.neighbors[node_idx]
                if action[neighbor] == 0
            ]

            if len(active_neighbors) == 0:
                action[node_idx] = 0

        active_mask = action == 0

        # --------------------------------------------------
        # 2. Redistribute sleeping-cell traffic.
        # --------------------------------------------------

        effective_load = np.zeros(
            self.num_nodes,
            dtype=np.float32
        )

        # Active cells initially handle their own demand.
        effective_load[active_mask] = traffic[active_mask]

        for node_idx in range(self.num_nodes):

            if active_mask[node_idx]:
                continue

            active_neighbors = [
                neighbor
                for neighbor in self.neighbors[node_idx]
                if active_mask[neighbor]
            ]

            # Defensive fallback.
            if len(active_neighbors) == 0:
                action[node_idx] = 0
                active_mask[node_idx] = True
                effective_load[node_idx] = traffic[node_idx]
                continue

            redistributed_load = (
                traffic[node_idx] / len(active_neighbors)
            )

            for neighbor in active_neighbors:
                effective_load[neighbor] += redistributed_load

        # --------------------------------------------------
        # 3. Calculate utilization and QoS estimates.
        # --------------------------------------------------

        utilization = (
            effective_load / np.maximum(capacity, 1e-6)
        )

        # Only active cells serve redistributed traffic.
        active_utilization = utilization[active_mask]

        # Deterministic approximation based on utilization.
        # These are simulation assumptions, not measured QoS.
        packet_loss = np.where(
            utilization < 0.8,
            0.005,
            np.where(utilization <= 1.0, 0.03, 0.08)
        )

        latency = 20 + 50 * utilization**3

        overload_fraction = (
            np.mean(active_utilization > 1.0)
            if active_utilization.size else 1.0
        )

        latency_violation_fraction = (
            np.mean(latency[active_mask] > 50)
            if active_mask.any() else 1.0
        )

        packet_loss_violation_fraction = (
            np.mean(packet_loss[active_mask] > 0.05)
            if active_mask.any() else 1.0
        )

        # --------------------------------------------------
        # 4. Calculate energy consumption.
        # --------------------------------------------------

        # Active power depends on load, capped at the
        # baseline power reference.
        active_power = (
            idle_power
            + (baseline_power - idle_power)
            * np.clip(utilization, 0, 1)
        )

        power = np.zeros(
            self.num_nodes,
            dtype=np.float32
        )

        power[action == 0] = active_power[action == 0]
        power[action == 1] = 0.40 * baseline_power[action == 1]
        power[action == 2] = 0.10 * baseline_power[action == 2]

        # One simulated step is one hour:
        # W / 1000 = kWh consumed in one hour.
        energy_kwh = float(power.sum() / 1000)

        # Counterfactual: all cells active under original demand.
        original_utilization = traffic / np.maximum(capacity, 1e-6)

        all_active_power = (
            idle_power
            + (baseline_power - idle_power)
            * np.clip(original_utilization, 0, 1)
        )

        baseline_energy_kwh = float(
            all_active_power.sum() / 1000
        )

        energy_savings_fraction = (
            (baseline_energy_kwh - energy_kwh)
            / max(baseline_energy_kwh, 1e-6)
        )

        # --------------------------------------------------
        # 5. Calculate reward.
        # --------------------------------------------------

        reward = (
            energy_savings_fraction
            - 2.0 * overload_fraction
            - 1.0 * latency_violation_fraction
            - 1.0 * packet_loss_violation_fraction
        )

        self.power_state = action.copy()

        info = {
            "energy_kwh": energy_kwh,
            "baseline_energy_kwh": baseline_energy_kwh,
            "energy_savings_fraction": energy_savings_fraction,
            "overload_fraction": float(overload_fraction),
            "latency_violation_fraction": float(
                latency_violation_fraction
            ),
            "packet_loss_violation_fraction": float(
                packet_loss_violation_fraction
            ),
            "active_cells": int(active_mask.sum()),
            "light_sleep_cells": int((action == 1).sum()),
            "deep_sleep_cells": int((action == 2).sum())
        }

        # Advance to the next simulated hour.
        self.t += 1

        terminated = self.t >= len(self.timestamps) - 1
        truncated = False

        if terminated:
            self.t = len(self.timestamps) - 1

        observation = self._get_observation()

        return (
            observation,
            float(reward),
            terminated,
            truncated,
            info
        )

Create the environment

In [19]:
env = EcoCellEnv(
    traffic_df=traffic_df,
    network_df=network_df,
    neighbors=neighbors
)

observation, info = env.reset(seed=42)

print("Observation shape:", observation.shape)
print("Observation space:", env.observation_space)

print("Action space:", env.action_space)
print("Number of possible actions per base station:",
      env.action_space.nvec[0])

print("First 12 observation values:", observation[:12])

Observation shape: (300,)
Observation space: Box(0.0, 1.0, (300,), float32)
Action space: MultiDiscrete([3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3 3 3 3 3 3])
Number of possible actions per base station: 3
First 12 observation values: [0.01164194 0.034      0.01164194 0.03309616 0.18827109 0.
 0.0109709  0.03       0.0109709  0.09013753 0.20964499 0.        ]


Test an all-active action

In [20]:
all_active_action = np.zeros(
    50,
    dtype=np.int64
)

observation, reward, terminated, truncated, info = (
    env.step(all_active_action)
)

print("Reward:", reward)
print("Terminated:", terminated)
print("\nStep information:")

for key, value in info.items():
    print(f"{key}: {value}")

Reward: 0.0
Terminated: False

Step information:
energy_kwh: 35.76417922973633
baseline_energy_kwh: 35.76417922973633
energy_savings_fraction: 0.0
overload_fraction: 0.0
latency_violation_fraction: 0.0
packet_loss_violation_fraction: 0.0
active_cells: 50
light_sleep_cells: 0
deep_sleep_cells: 0


Test a Sleep-mode action

In [21]:
observation, info = env.reset(seed=42)

test_action = np.zeros(
    50,
    dtype=np.int64
)

# Request light sleep for BS00 and BS01.
test_action[node_to_idx["BS00"]] = 1
test_action[node_to_idx["BS01"]] = 1

# Request deep sleep for BS02.
test_action[node_to_idx["BS02"]] = 2

observation, reward, terminated, truncated, info = (
    env.step(test_action)
)

print("Reward:", reward)

for key, value in info.items():
    print(f"{key}: {value}")

Reward: 0.025581724018211557
energy_kwh: 34.84926986694336
baseline_energy_kwh: 35.76417922973633
energy_savings_fraction: 0.025581724018211557
overload_fraction: 0.0
latency_violation_fraction: 0.0
packet_loss_violation_fraction: 0.0
active_cells: 47
light_sleep_cells: 2
deep_sleep_cells: 1


Step 9 — Evaluate both policies over the full episode

In [26]:

def evaluate_policy(env, policy_name, policy_fn):
    observation, _ = env.reset(seed=42)

    records = []
    terminated = False
    truncated = False

    while not (terminated or truncated):
        action = policy_fn(observation)

        observation, reward, terminated, truncated, info = env.step(action)

        records.append({
            "reward": reward,
            **info
        })

    results = pd.DataFrame(records)

    summary = {
        "policy": policy_name,
        "hours_evaluated": len(results),
        "mean_energy_kwh": results["energy_kwh"].mean(),
        "mean_energy_savings_pct": (
            results["energy_savings_fraction"].mean() * 100
        ),
        "mean_reward": results["reward"].mean(),
        "mean_active_cells": results["active_cells"].mean(),
        "mean_light_sleep_cells": results["light_sleep_cells"].mean(),
        "mean_deep_sleep_cells": results["deep_sleep_cells"].mean(),
        "hours_with_overload": (
            results["overload_fraction"] > 0
        ).sum(),
        "hours_with_latency_violations": (
            results["latency_violation_fraction"] > 0
        ).sum(),
        "hours_with_packet_loss_violations": (
            results["packet_loss_violation_fraction"] > 0
        ).sum()
    }

    return summary, results


# Policy A: keep every base station active.
def all_active_policy(observation):
    return np.zeros(50, dtype=np.int64)


# Policy B: light-sleep cells with low observed utilization.
# Feature 0 is utilization divided by 3, clipped to [0, 1].
def low_utilization_policy(observation):
    node_features = observation.reshape(50, 6)

    action = np.zeros(50, dtype=np.int64)
    low_utilization = node_features[:, 0] < (0.06 / 3)

    action[low_utilization] = 1  # Light sleep
    return action


baseline_summary, baseline_results = evaluate_policy(
    env, "All-active baseline", all_active_policy
)

heuristic_summary, heuristic_results = evaluate_policy(
    env, "Low-utilization heuristic", low_utilization_policy
)

comparison = pd.DataFrame([
    baseline_summary,
    heuristic_summary
])

display(comparison.round(3))

,policy,hours_evaluated,mean_energy_kwh,mean_energy_savings_pct,mean_reward,mean_active_cells,mean_light_sleep_cells,mean_deep_sleep_cells,hours_with_overload,hours_with_latency_violations,hours_with_packet_loss_violations
0,All-active baseline,719,37.354,0.000,0.000,50.000,0.000,0.0,0,0,0
1,Low-utilization heuristic,719,35.116,6.243,0.062,40.146,9.854,0.0,0,0,0


Correct the observation features

In [27]:

def _get_observation(self):
    traffic = self.data["traffic_mbps"][self.t]
    capacity = self.data["capacity_mbps"][self.t]
    users = self.data["active_users"][self.t]
    packet_loss = self.data["packet_loss_rate"][self.t]
    latency = self.data["latency_ms"][self.t]

    utilization = traffic / np.maximum(capacity, 1e-6)

    features = np.column_stack([
        np.clip(traffic / 1000.0, 0, 1),       # Traffic volume
        np.clip(users / 500.0, 0, 1),          # Active users
        np.clip(utilization / 3.0, 0, 1),      # Utilization
        np.clip(packet_loss / 0.10, 0, 1),     # Packet loss
        np.clip(latency / 100.0, 0, 1),        # Latency
        self.power_state / 2.0                 # Power state
    ])

    return features.astype(np.float32).flatten()


In [28]:
env = EcoCellEnv(
    traffic_df=traffic_df,
    network_df=network_df,
    neighbors=neighbors
)

observation, info = env.reset(seed=42)

print("Observation shape:", observation.shape)
print("First 12 observation values:", observation[:12])

Observation shape: (300,)
First 12 observation values: [0.01164194 0.034      0.01164194 0.03309616 0.18827109 0.
 0.0109709  0.03       0.0109709  0.09013753 0.20964499 0.        ]


Step 12 — Update the heuristic

In [29]:
def low_utilization_policy(observation):
    node_features = observation.reshape(50, 6)

    action = np.zeros(50, dtype=np.int64)

    low_utilization = node_features[:, 2] < (0.06 / 3)
    action[low_utilization] = 1

    return action